# Exercício — Pipeline RAG local com PDFs, Ollama, FAISS e LangChain

Este notebook implementa um pipeline RAG completo usando:

- carregamento exclusivo de PDFs a partir de um diretório;
- chunking com splitter baseado em tokens do Hugging Face;
- embeddings locais via Ollama com `bge-m3`;
- armazenamento vetorial local com FAISS;
- retriever sobre o VectorStore;
- chamada ao LLM local via Ollama, por exemplo `gemma3:4b`;
- cadeia RAG com LangChain;
- reescrita de query;
- MultiQuery Retriever;
- técnica HyDE para geração de documento hipotético;
- logs opcionais com LangSmith.

> Observação: o enunciado menciona “Files”, mas no contexto de banco vetorial local estou considerando **FAISS**.


## 1. Instalação das dependências

In [ ]:
# Se necessário, descomente e execute:
# %pip install -U langchain langchain-community langchain-ollama langchain-text-splitters
# %pip install -U pypdf faiss-cpu transformers sentencepiece langsmith

## 2. Preparação do Ollama

Antes de rodar o notebook, execute no terminal:

```bash
ollama pull bge-m3
ollama pull gemma3:4b
ollama serve
```

Caso o modelo `gemma3:4b` não esteja disponível no seu ambiente, ajuste a variável `LLM_MODEL` para outro modelo local instalado, por exemplo:

```python
LLM_MODEL = "llama3.2:3b"
```


In [ ]:
import os
from pathlib import Path

# Diretório onde ficarão os PDFs
PDF_DIR = Path("./pdfs")

# Diretório local onde o índice vetorial será salvo
VECTOR_DB_DIR = Path("./vectorstore_faiss")

# Modelos locais via Ollama
EMBEDDING_MODEL = "bge-m3"
LLM_MODEL = "gemma3:4b"

PDF_DIR.mkdir(exist_ok=True)
VECTOR_DB_DIR.mkdir(exist_ok=True)

print(f"Diretório dos PDFs: {PDF_DIR.resolve()}")
print(f"Diretório do VectorStore: {VECTOR_DB_DIR.resolve()}")

## 3. Configuração opcional do LangSmith

Se quiser comparar os logs entre consulta original, query rewriter, MultiQuery e HyDE, configure as variáveis abaixo.

Se não for usar LangSmith, mantenha desativado.


In [ ]:
# Ative apenas se possuir chave do LangSmith.
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "sua-chave-aqui"
# os.environ["LANGSMITH_PROJECT"] = "rag-pdfs-local-exercicio"

print("LangSmith configurado?", os.environ.get("LANGSMITH_TRACING", "false"))

## 4. Carregamento exclusivo de PDFs com DocumentLoader

Aqui usamos `DirectoryLoader` apontando para `*.pdf`, com `PyPDFLoader` como loader específico.

Coloque seus arquivos PDF dentro da pasta `./pdfs`.


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

loader = DirectoryLoader(
    path=str(PDF_DIR),
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True,
)

documents = loader.load()

print(f"Quantidade de páginas/documentos carregados: {len(documents)}")

if documents:
    print("\nExemplo de metadados do primeiro documento:")
    print(documents[0].metadata)
    print("\nPrévia do conteúdo:")
    print(documents[0].page_content[:500])
else:
    print("Nenhum PDF encontrado. Adicione arquivos na pasta ./pdfs e execute novamente.")

## 5. Splitter baseado em tokens do Hugging Face

Nesta etapa, usamos um tokenizer do Hugging Face para dividir o texto em chunks por tokens.

O tamanho e o overlap podem ser ajustados conforme o tipo de documento.


In [ ]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

TOKENIZER_NAME = "BAAI/bge-m3"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(documents)

print(f"Total de documentos/páginas originais: {len(documents)}")
print(f"Total de chunks gerados: {len(chunks)}")

if chunks:
    print("\nMetadados do primeiro chunk:")
    print(chunks[0].metadata)
    print("\nTamanho aproximado do primeiro chunk em caracteres:")
    print(len(chunks[0].page_content))
    print("\nPrévia do primeiro chunk:")
    print(chunks[0].page_content[:700])

## 6. Embeddings locais com BGE-M3 via Ollama

Nesta etapa, criamos o modelo de embeddings local.

Garanta que o Ollama esteja rodando e que o modelo tenha sido baixado:

```bash
ollama pull bge-m3
```


In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

# Teste rápido do embedding
teste_vetor = embeddings.embed_query("Teste de geração de embeddings com BGE-M3 via Ollama.")
print(f"Dimensão do vetor gerado: {len(teste_vetor)}")
print(teste_vetor[:5])

## 7. Armazenamento dos chunks no VectorStore local com FAISS

O índice será criado localmente e salvo em disco.


In [ ]:
from langchain_community.vectorstores import FAISS

if not chunks:
    raise ValueError("Nenhum chunk foi gerado. Verifique se existem PDFs válidos na pasta ./pdfs.")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

vectorstore.save_local(str(VECTOR_DB_DIR))

print(f"VectorStore salvo em: {VECTOR_DB_DIR.resolve()}")
print(f"Total de chunks indexados: {len(chunks)}")

## 8. Carregando o VectorStore salvo

Esta etapa simula o uso posterior do índice já persistido.


In [ ]:
vectorstore = FAISS.load_local(
    folder_path=str(VECTOR_DB_DIR),
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print("Retriever criado com sucesso.")

## 9. Teste simples do retriever

Faça uma consulta relacionada aos PDFs carregados.


In [ ]:
consulta = "Qual é o tema principal dos documentos?"

docs_retornados = retriever.invoke(consulta)

print(f"Documentos retornados: {len(docs_retornados)}")

for i, doc in enumerate(docs_retornados, start=1):
    print(f"\n--- Documento {i} ---")
    print("Fonte:", doc.metadata.get("source"))
    print("Página:", doc.metadata.get("page"))
    print(doc.page_content[:500])

## 10. Teste do LLM local via Ollama

Garanta que o modelo esteja disponível localmente:

```bash
ollama pull gemma3:4b
```


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0.2,
)

resposta_teste = llm.invoke("Responda em uma frase: o que é RAG?")
print(resposta_teste.content)

## 11. Cadeia RAG básica

Agora integramos:

- retriever;
- prompt;
- modelo local;
- parser de saída.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[Fonte: {doc.metadata.get('source')} | Página: {doc.metadata.get('page')}]\n{doc.page_content}"
        for doc in docs
    )

rag_prompt = ChatPromptTemplate.from_template(
    """Você é um assistente técnico. Responda com base apenas no contexto fornecido.

Contexto:
{context}

Pergunta:
{question}

Resposta em português, objetiva e bem estruturada:
"""
)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

resposta = rag_chain.invoke(consulta)
print(resposta)

## 12. Técnica de reescrita de query

A ideia aqui é usar um modelo menor ou mais barato para transformar a pergunta original em uma consulta mais clara para recuperação semântica.

Neste exemplo, uso o mesmo Ollama por simplicidade, mas você pode configurar outro modelo menor se tiver instalado.


In [ ]:
QUERY_REWRITER_MODEL = "llama3.2:3b"  # Mesmo modelo do LLM principal

query_rewriter_llm = ChatOllama(
    model=QUERY_REWRITER_MODEL,
    temperature=0,
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """Reescreva a pergunta abaixo para melhorar a busca semântica em documentos PDF.

Regras:
- mantenha o sentido original;
- não responda à pergunta;
- gere apenas uma versão refinada da consulta;
- remova ambiguidades;
- preserve termos técnicos importantes.

Pergunta original:
{question}

Consulta refinada:
"""
)

query_rewriter_chain = rewrite_prompt | query_rewriter_llm | StrOutputParser()

consulta_refinada = query_rewriter_chain.invoke({"question": consulta})

print("Consulta original:")
print(consulta)

print("\nConsulta refinada:")
print(consulta_refinada)

## 13. Comparação: consulta original vs consulta reescrita

Aqui comparamos os documentos retornados por cada consulta.


In [ ]:
docs_original = retriever.invoke(consulta)
docs_rewritten = retriever.invoke(consulta_refinada)

print("=== Resultado com consulta original ===")
for i, doc in enumerate(docs_original, start=1):
    print(f"\n[{i}] Fonte: {doc.metadata.get('source')} | Página: {doc.metadata.get('page')}")
    print(doc.page_content[:350])

print("\n\n=== Resultado com consulta refinada ===")
for i, doc in enumerate(docs_rewritten, start=1):
    print(f"\n[{i}] Fonte: {doc.metadata.get('source')} | Página: {doc.metadata.get('page')}")
    print(doc.page_content[:350])

## 14. RAG usando a consulta reescrita

Agora usamos a consulta refinada para alimentar a cadeia RAG.


In [ ]:
resposta_rewritten = rag_chain.invoke(consulta_refinada)

print("Resposta usando consulta refinada:")
print(resposta_rewritten)

## 15. MultiQuery Retriever

O MultiQuery gera variações da pergunta com o LLM, busca documentos para cada variação e consolida os resultados.

Isso ajuda quando a pergunta original é curta, ambígua ou usa palavras diferentes das existentes no documento.


In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm,
)

docs_multiquery = multiquery_retriever.invoke(consulta)

print(f"Documentos consolidados pelo MultiQuery: {len(docs_multiquery)}")

for i, doc in enumerate(docs_multiquery, start=1):
    print(f"\n[{i}] Fonte: {doc.metadata.get('source')} | Página: {doc.metadata.get('page')}")
    print(doc.page_content[:350])

## 16. Cadeia RAG com MultiQuery Retriever

Aqui trocamos apenas o retriever.


In [ ]:
rag_chain_multiquery = (
    {
        "context": multiquery_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

resposta_multiquery = rag_chain_multiquery.invoke(consulta)

print(resposta_multiquery)

## 17. Técnica HyDE

HyDE significa Hypothetical Document Embeddings.

Em vez de buscar diretamente pela pergunta, primeiro pedimos ao LLM para gerar um parágrafo hipotético que provavelmente responderia à pergunta. Depois usamos esse parágrafo como consulta semântica no VectorStore.


In [ ]:
hyde_prompt = ChatPromptTemplate.from_template(
    """Gere um parágrafo hipotético, técnico e objetivo, que poderia aparecer em um documento PDF respondendo à pergunta abaixo.

Não diga que é hipotético.
Não use introduções.
Escreva apenas o parágrafo.

Pergunta:
{question}

Parágrafo:
"""
)

hyde_chain = hyde_prompt | llm | StrOutputParser()

hyde_document = hyde_chain.invoke({"question": consulta})

print("Documento hipotético gerado pelo HyDE:")
print(hyde_document)

In [ ]:
docs_hyde = retriever.invoke(hyde_document)

print(f"Documentos retornados via HyDE: {len(docs_hyde)}")

for i, doc in enumerate(docs_hyde, start=1):
    print(f"\n[{i}] Fonte: {doc.metadata.get('source')} | Página: {doc.metadata.get('page')}")
    print(doc.page_content[:350])

## 18. Cadeia RAG com HyDE

Aqui o contexto vem dos documentos recuperados a partir do parágrafo hipotético, mas a resposta final continua respondendo à pergunta original.


In [ ]:
def hyde_retrieve(question: str):
    hypothetical_doc = hyde_chain.invoke({"question": question})
    return retriever.invoke(hypothetical_doc)

rag_chain_hyde = (
    {
        "context": hyde_retrieve | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

resposta_hyde = rag_chain_hyde.invoke(consulta)

print(resposta_hyde)

## 19. Comparação final dos resultados

A comparação ideal deve considerar:

- os documentos recuperados;
- a relevância dos trechos;
- a qualidade da resposta;
- presença ou ausência de alucinação;
- rastreabilidade da fonte;
- comportamento no LangSmith, se habilitado.


In [ ]:
print("=" * 80)
print("CONSULTA ORIGINAL")
print("=" * 80)
print(resposta)

print("\n" + "=" * 80)
print("CONSULTA REESCRITA")
print("=" * 80)
print(resposta_rewritten)

print("\n" + "=" * 80)
print("MULTIQUERY")
print("=" * 80)
print(resposta_multiquery)

print("\n" + "=" * 80)
print("HYDE")
print("=" * 80)
print(resposta_hyde)

## 20. Observações para entrega da atividade

Neste notebook foi implementado um pipeline RAG local completo para leitura de PDFs, divisão em chunks por tokens, geração de embeddings locais via Ollama, indexação em FAISS e recuperação semântica. Também foram testadas técnicas de melhoria de busca, como reescrita de query, MultiQuery Retriever e HyDE.

A proposta permite comparar a recuperação original com abordagens mais avançadas, observando se os documentos retornados são mais relevantes e se a resposta final fica mais precisa. Com LangSmith habilitado, é possível analisar os logs de cada etapa e visualizar claramente a diferença entre cada técnica.
